In [ ]:
import os
from time import perf_counter
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR
from torch.nn.utils import vector_to_parameters, parameters_to_vector
from scipy.optimize import minimize
from scipy.linalg import cholesky, LinAlgError



results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

SAVE_MODEL = True
MODEL_PATH = os.path.join(results_dir, "model_final.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_dtype(torch.float64)

seed = 4
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Network
layers = 4
neurons = 30
output_dim = 1
kmax = 1

# PDE parameters
a1 = 1.0
a2 = 4.0
k = 1.0

# Domain
x0 = -1.0
y0 = -1.0
Lx = 2.0
Ly = 2.0
xf = x0 + Lx
yf = y0 + Ly

# Collocation 
Nint = 15000
Nchange = 200
k1 = 1.0
k2 = 0.0
rad_args = (k1, k2)

# ADAM warmup
Nepochs_ADAM = 1000     
Nprint_adam = 10
lr0 = 5e-3
b1 = 0.99
b2 = 0.999
epsilon = 1e-20
decay_steps = 1000
decay_rate = 0.98

# BFGS / SSBFGS / SSBroyden training
Nbfgs = 100000
Nprint_bfgs = 100
method = "BFGS"
method_bfgs = "SSBroyden2"  
initial_scale = False

use_sqrt = False
use_log = False

TOTAL_BUDGET_SEC = 20000.0
deadline = perf_counter() + TOTAL_BUDGET_SEC


def time_up():
    return perf_counter() >= deadline


class PeriodicC0Layer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, inputs):
        x = inputs[:, 0:1]
        y = inputs[:, 1:2]

        ks = torch.arange(
            1,
            kmax + 1,
            device=inputs.device,
            dtype=torch.get_default_dtype(),
        )

        xper = 2 * torch.pi * (x @ ks[None, :]) / Lx
        yper = 2 * torch.pi * (y @ ks[None, :]) / Ly

        xcos, xsin = torch.cos(xper), torch.sin(xper)
        ycos, ysin = torch.cos(yper), torch.sin(yper)

        return torch.cat([xcos, xsin, ycos, ysin], dim=1)


class ModelTorch(nn.Module):
    def __init__(self, layer_dims):
        super().__init__()
        self.per = PeriodicC0Layer()

        modules = []
        in_dim = 4 * kmax
        modules.append(nn.Linear(in_dim, layer_dims[1]))
        modules.append(nn.Tanh())

        prev = layer_dims[1]
        for width in layer_dims[2:-1]:
            modules.append(nn.Linear(prev, width))
            modules.append(nn.Tanh())
            prev = width

        modules.append(nn.Linear(prev, layer_dims[-1]))
        self.net = nn.Sequential(*modules)

    def forward(self, X_input):
        return self.net(self.per(X_input))


def generate_inputs(n_points, device):
    x = (xf - x0) * torch.rand(n_points, 1, device=device) + x0
    y = (yf - y0) * torch.rand(n_points, 1, device=device) + y0
    return torch.cat([x, y], dim=1)


def output(model, X):
    return model(X)[:, 0:1]


def get_results(model, X):
    with torch.enable_grad():
        X = X.detach().clone().requires_grad_(True)
        u = output(model, X)

        du_dX = torch.autograd.grad(u, X, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True,)[0]
        u_x = du_dX[:, 0:1]
        u_y = du_dX[:, 1:2]

        u_xx = torch.autograd.grad(u_x, X, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True,)[0][:, 0:1]
        u_yy = torch.autograd.grad(u_y, X, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True,)[0][:, 1:2]

        x = X[:, 0:1]
        y = X[:, 1:2]

        exact_term = torch.sin(a1 * torch.pi * x) * torch.sin(a2 * torch.pi * y)
        q = -((a1 * torch.pi) ** 2) * exact_term \
            - ((a2 * torch.pi) ** 2) * exact_term \
            + (k ** 2) * exact_term

        residual = u_xx + u_yy + (k ** 2) * u - q
        return u, residual


loss_function = nn.MSELoss()


def loss(residual):
    return loss_function(residual, torch.zeros_like(residual))


def adaptive_rad(model, n_points, rad_args, n_test=50000):
    Xtest = generate_inputs(n_test, next(model.parameters()).device)
    _, residual = get_results(model, Xtest)

    y_abs = torch.abs(residual.detach()).reshape(-1)
    k1_val, k2_val = rad_args
    eps = 1e-12

    weights = torch.pow(y_abs + eps, k1_val)
    weights = weights / torch.clamp(weights.mean(), min=eps) + float(k2_val)
    probs = weights / torch.clamp(weights.sum(), min=eps)
    probs = torch.clamp(probs, min=0)
    probs = probs / torch.clamp(probs.sum(), min=eps)

    n_choose = min(n_points, Xtest.shape[0])
    idx = torch.multinomial(probs, num_samples=n_choose, replacement=False)
    return Xtest.index_select(0, idx).detach()



X = generate_inputs(Nint, device)

layer_dims = [X.shape[1]] + [neurons] * layers + [output_dim]
model = ModelTorch(layer_dims).to(device)

print(f"device: {device}")
print(f"method_bfgs: {method_bfgs}")


if Nepochs_ADAM > 0:
    optimizer = Adam(model.parameters(), lr=lr0, betas=(b1, b2), eps=epsilon)
    scheduler = LambdaLR(
        optimizer,
        lr_lambda=lambda step: decay_rate ** (step / decay_steps),)

    for epoch in range(Nepochs_ADAM):
        if time_up():
            print("[ADAM] time budget reached; stopping ADAM warmup.")
            break

        if (epoch + 1) % Nchange == 0:
            X = adaptive_rad(model, Nint, rad_args)

        optimizer.zero_grad(set_to_none=True)
        X_batch = X.detach().clone().to(device).requires_grad_(True)
        _, residual = get_results(model, X_batch)
        loss_value = loss(residual)
        loss_value.backward()
        optimizer.step()
        scheduler.step()

        if (epoch + 1) % Nprint_adam == 0:
            print(f"[ADAM] epoch={epoch + 1} loss={loss_value.item():.6e}")
else:
    print("[ADAM] skipped")



initial_weights = parameters_to_vector([p.detach() for p in model.parameters()]).cpu().numpy()


class TimeBudgetExceeded(Exception):
    pass


def loss_and_gradient(weights, model, X, use_sqrt=False, use_log=False):
    p0 = next(model.parameters())
    w_tensor = torch.as_tensor(weights, dtype=p0.dtype, device=p0.device)

    with torch.no_grad():
        vector_to_parameters(w_tensor, model.parameters())

    X_in = X.detach().clone().requires_grad_(True)
    _, residual = get_results(model, X_in)
    base_loss = loss(residual)

    if use_sqrt:
        objective = torch.sqrt(base_loss)
    elif use_log:
        objective = torch.log(base_loss)
    else:
        objective = base_loss

    params = [p for p in model.parameters() if p.requires_grad]
    grads = torch.autograd.grad(objective, params)
    grads_flat = torch.cat([g.reshape(-1) for g in grads]).detach().cpu().numpy()

    return float(objective.detach().cpu().item()), grads_flat


iteration = 0
bfgs_t0 = perf_counter()


def callback(*, intermediate_result):
    global iteration

    if time_up():
        raise TimeBudgetExceeded("Global time budget exceeded inside BFGS callback.")

    iteration += 1
    if iteration == 1 or iteration % Nprint_bfgs == 0:
        elapsed = perf_counter() - bfgs_t0
        print(
            f"[{method_bfgs}] iter={iteration} "
            f"loss={float(intermediate_result.fun):.6e} "
            f"elapsed={elapsed:.2f}s")


def make_bfgs_options(hess_inv0):
    return {
        "maxiter": Nchange,
        "gtol": 0,
        "hess_inv0": hess_inv0,
        "method_bfgs": method_bfgs,
        "initial_scale": initial_scale,}


H0 = np.eye(len(initial_weights), dtype=np.float64)
options = make_bfgs_options(H0)

while iteration < Nbfgs:
    if time_up():
        print("[BFGS] time budget reached; stopping BFGS loop.")
        break

    try:
        result = minimize(
            loss_and_gradient,
            initial_weights,
            args=(model, X, use_sqrt, use_log),
            method=method,
            jac=True,
            options=options,
            tol=0,
            callback=callback,)
    except TimeBudgetExceeded:
        print("[BFGS] time budget reached; aborting current SciPy minimize call.")
        break

    initial_weights = result.x
    with torch.no_grad():
        vector_to_parameters( torch.as_tensor(initial_weights, dtype=next(model.parameters()).dtype, device=next(model.parameters()).device,),
            model.parameters(),)

    H0 = getattr(result, "hess_inv", None)
    if H0 is None:
        H0 = np.eye(len(initial_weights), dtype=np.float64)
    else:
        H0 = 0.5 * (H0 + H0.T)
        try:
            cholesky(H0)
        except LinAlgError:
            H0 = np.eye(len(initial_weights), dtype=np.float64)

    options = make_bfgs_options(H0)
    X = adaptive_rad(model, Nint, rad_args)


if SAVE_MODEL:
    torch.save(model.state_dict(), MODEL_PATH)
    print(f"[save] model saved to {MODEL_PATH}")
